In [1]:
import pandas as pd

df = pd.read_csv("hotel_bookings.csv")

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [2]:
print(df.shape)
df.head(3)

(119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02


## 1. The data

**What it is:** Booking records from two hotels, one city hotel (79,330 rows) and one resort hotel (40,060 rows), with arrivals from July 2015 to August 2017. The data has 119,390 rows and 32 columns.

**Where it comes from:** Originally from the article "Hotel Booking Demand Datasets" (Antonio, Almeida & Nunes, *Data in Brief*, vol. 22, 2019). Downloaded and cleaned by Thomas Mock and Antoine Bichat for #TidyTuesday (2020), and obtained from Kaggle.

**What one row represents:** One booking. It records when the booking was made, the arrival date, the length of stay, the guests, the price and whether it was cancelled (`is_canceled`).

## 2. Data quality (the five C's)

### Clean

In [6]:
df.head(5).T

,0,1,2,3,4
hotel,Resort Hotel,Resort Hotel,Resort Hotel,Resort Hotel,Resort Hotel
is_canceled,0,0,0,0,0
lead_time,342,737,7,13,14
arrival_date_year,2015,2015,2015,2015,2015
arrival_date_month,July,July,July,July,July
arrival_date_week_number,27,27,27,27,27
arrival_date_day_of_month,1,1,1,1,1
stays_in_weekend_nights,0,0,0,0,0
stays_in_week_nights,0,0,1,1,2
adults,2,2,1,1,2


In [3]:
nulos = df.isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
print((nulos / len(df) * 100).round(3))
print(nulos)

print("Copias sobrantes:", df.duplicated().sum())
print("% del total:", round(df.duplicated().mean() * 100, 1))

company     94.307
agent       13.686
country      0.409
children     0.003
dtype: float64
company     112593
agent        16340
country        488
children         4
dtype: int64
Copias sobrantes: 31994
% del total: 26.8


**Finding:** Four columns have missing values: `company` (94.3%), `agent` (13.7%), `country` (0.4%) and `children` (4 rows). In addition, 31,994 rows (26.8%) are exact copies of another row. The data has no booking ID, so we cannot tell whether they are errors or real bookings.

We keep them as real bookings: small repeats can be different guests with the same details, and large repeats (up to 180 identical rows) look like block bookings, mostly from groups and tour operators. This is an assumption, because we cannot verify it without an ID.

### Consistent

In [8]:
for col in ["meal", "market_segment", "distribution_channel"]:
    print(df[col].value_counts(), "\n")

print(df.loc[df["country"].isin(["CN", "CHN"]), "country"].value_counts(), "\n")

print(pd.crosstab(df["is_canceled"], df["reservation_status"]))

meal
BB           92310
HB           14463
SC           10650
Undefined     1169
FB             798
Name: count, dtype: int64 

market_segment
Online TA        56477
Offline TA/TO    24219
Groups           19811
Direct           12606
Corporate         5295
Complementary      743
Aviation           237
Undefined            2
Name: count, dtype: int64 

distribution_channel
TA/TO        97870
Direct       14645
Corporate     6677
GDS            193
Undefined        5
Name: count, dtype: int64 

country
CN     1279
CHN     999
Name: count, dtype: int64 

reservation_status  Canceled  Check-Out  No-Show
is_canceled                                     
0                          0      75166        0
1                      43017          0     1207


**Finding:** Text labels have no case or spacing problems, but we found three inconsistencies. In `meal`, "Undefined" (1,169 rows) and "SC" (10,650 rows) mean the same thing according to the data dictionary: no meal package. `market_segment` (2 rows) and `distribution_channel` (5 rows) contain an undocumented "Undefined" label. China appears under two codes in `country`: `CN` (1,279 rows) and `CHN` (999 rows).

`is_canceled` fully agrees with `reservation_status`. The target includes 1,207 no-shows, so "cancelled" means "the guest did not stay".

### Current

In [9]:
fechas = pd.to_datetime(
    df["arrival_date_year"].astype(str) + "-"
    + df["arrival_date_month"] + "-"
    + df["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d"
)

print("Primera llegada:", fechas.min().date())
print("Última llegada:", fechas.max().date())

print(df.groupby("arrival_date_year")["is_canceled"].agg(["size", "mean"]).round(3))

Primera llegada: 2015-07-01
Última llegada: 2017-08-31
                    size   mean
arrival_date_year              
2015               21996  0.370
2016               56707  0.359
2017               40687  0.387


**Finding:** Arrivals run from 1 July 2015 to 31 August 2017, and only 2016 is a complete year. The data is about nine years old and predates COVID-19, so booking behaviour today may be different. The cancellation rate is similar across years (35.9% to 38.7%), although 2015 and 2017 are partial years.

Any recommendation assumes that these patterns still hold, so the model should be checked on recent bookings before the hotel uses it.

### Correct

In [10]:
print(df[["lead_time", "adults", "children", "babies", "adr"]].describe().T[["min", "max"]])

huespedes = df["adults"] + df["children"].fillna(0) + df["babies"]
noches = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]

sin_huespedes = huespedes == 0
sin_noches = noches == 0
precio_negativo = df["adr"] < 0
precio_extremo = df["adr"] > 1000

print("Sin huéspedes:", sin_huespedes.sum())
print("Cero noches:", sin_noches.sum())
print("Precio negativo:", precio_negativo.sum())
print("Precio sobre 1000:", precio_extremo.sum())
print("Con algún defecto:", (sin_huespedes | sin_noches | precio_negativo | precio_extremo).sum())

print("Precio cero:", (df["adr"] == 0).sum())
print("10 adultos o más:", (df["adults"] >= 10).sum())

            min     max
lead_time  0.00   737.0
adults     0.00    55.0
children   0.00    10.0
babies     0.00    10.0
adr       -6.38  5400.0
Sin huéspedes: 180
Cero noches: 715
Precio negativo: 1
Precio sobre 1000: 1
Con algún defecto: 827
Precio cero: 1959
10 adultos o más: 13


**Finding:** 827 rows (0.7%) have impossible or highly doubtful values: 180 bookings with no guests, 715 with zero nights, one negative price (-6.38) and one price of 5,400 when the next highest is 510. Some rows have more than one defect. We will remove them during data preparation.

We keep the 1,959 bookings with a price of 0, because 680 of them are complimentary stays and a price of 0 is not impossible. We also keep the 13 bookings with 10 or more adults, because they could be groups. All 13 were cancelled.

### Comprehensive

In [11]:
print(df.groupby("hotel")["is_canceled"].agg(["size", "mean"]).round(3), "\n")
print(df["market_segment"].value_counts(normalize=True).round(3).head(4), "\n")
print(df["country"].value_counts(normalize=True).round(3).head(3), "\n")
print("Huéspedes repetidos:", round(df["is_repeated_guest"].mean(), 3))

               size   mean
hotel                     
City Hotel    79330  0.417
Resort Hotel  40060  0.278 

market_segment
Online TA        0.473
Offline TA/TO    0.203
Groups           0.166
Direct           0.106
Name: proportion, dtype: float64 

country
PRT    0.409
GBR    0.102
FRA    0.088
Name: proportion, dtype: float64 

Huéspedes repetidos: 0.032


**Finding:** The data covers only two hotels, and they behave differently: 41.7% of City Hotel bookings and 27.8% of Resort Hotel bookings were cancelled. Results may not apply to other hotels. Online travel agents account for 47.3% of bookings, Portugal for 40.9%, and only 3.2% of guests are repeat guests.

The data does not include a booking ID, hotel costs, cancellation reasons, competitor prices or whether a cancelled room was resold. The last gap matters most: we cannot observe the money actually lost per cancellation, so any business estimate will need a stated assumption.